In [36]:
from typing import TypedDict, List
from langgraph.graph import StateGraph

In [37]:
class AgentState(TypedDict):
    name: str
    age: str
    skills: List[str]
    result: str

In [38]:
def first_node_state(state: AgentState) -> AgentState:
    """ Personalizes the name field with a greeting """
    state["result"] = f"{state['name']}, welcome to the system! "
    return state

def second_node_state(state: AgentState) -> AgentState:
    """ Describes the user's age """
    state["result"] += f"You are {state['age']} years old! "
    return state

def third_node_state(state: AgentState) -> AgentState:
    """ Lists the user's skills in a formatted string with comma """
    skills = state.get("skills", [])
    
    if not skills:
        skills_str = "none"
    elif len(skills) == 1:
        skills_str = skills[0]
    elif len(skills) == 2:
        skills_str = f"{skills[0]} and {skills[1]}"
    else:
        skills_str = ", ".join(skills[:-1]) + f", and {skills[-1]}"
        
    state["result"] += f"You have skills in: {skills_str}"
    return state

In [39]:
graph = StateGraph(AgentState)

START_NODE = "first_node"
MIDDLE_NODE = "second_node"
END_NODE = "third_node"

graph.add_node(START_NODE, first_node_state)
graph.add_node(MIDDLE_NODE, second_node_state)
graph.add_node(END_NODE, third_node_state)

graph.set_entry_point(START_NODE)
graph.add_edge(START_NODE, MIDDLE_NODE)
graph.add_edge(MIDDLE_NODE, END_NODE)
graph.set_finish_point(END_NODE)

app = graph.compile()

In [40]:
input_data = {
    "name": "David",
    "age": "32",
    "skills": ["Python", "Machine Learning", "LangGraph", "Cloud Computing", "Cybersecurity"]
}

output_state = app.invoke(input_data)
output_state["result"]

'David, welcome to the system! You are 32 years old! You have skills in: Python, Machine Learning, LangGraph, Cloud Computing, and Cybersecurity'